In [1]:
# %%
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd

import mne

In [2]:
# %%
PROJECT_ROOT = Path("..").resolve()

DATA_ROOT = PROJECT_ROOT / "data"
SPONT_PATH = DATA_ROOT / "raw" / "spontaneous"

DERIVED_ROOT = DATA_ROOT / "derived"

MANIFEST_PATH = DERIVED_ROOT / "manifests" / "manifest_spontaneous_validated.csv"
FEATURE_DIR = DERIVED_ROOT / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# NEW: epoch-level output
FEATURES_A_EPOCH_PATH = FEATURE_DIR / "features_A_bandpower_epochwise.csv"

# Minimal frequency bands for bandpower features (Hz)
BANDS_HZ = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
}

# Minimal epoch rejection. Peak-to-peak threshold in microvolts.
REJECT_PTP_UV = 250.0

# Eyes condition for the minimal design
EYES_KEEP = "closed"

print("Manifest:", MANIFEST_PATH)
print("Output:", FEATURES_A_EPOCH_PATH)

Manifest: /Users/I743312/Documents/ketamine project/data/derived/manifests/manifest_spontaneous_validated.csv
Output: /Users/I743312/Documents/ketamine project/data/derived/features/features_A_bandpower_epochwise.csv


In [3]:
# %%
# ============================================
# Section 2. Load manifest and select EC-only rows
# ============================================

manifest = pd.read_csv(MANIFEST_PATH)

required_cols = ["subject_id", "file_path", "eyes", "drug", "recording_number"]
missing = [c for c in required_cols if c not in manifest.columns]
assert len(missing) == 0, f"Manifest missing columns: {missing}"

df = manifest[manifest["eyes"] == EYES_KEEP].copy()
df = df.sort_values(["subject_id", "recording_number"]).reset_index(drop=True)

print("Rows (recordings) in EC-only subset:", len(df))
print("Subjects in EC-only subset:", df["subject_id"].nunique())
print("Counts by drug:")
display(df.groupby("drug").size().rename("n_recordings"))
df.head(10)

Rows (recordings) in EC-only subset: 20
Subjects in EC-only subset: 10
Counts by drug:


drug
awake       10
ketamine    10
Name: n_recordings, dtype: int64

,subject_id,date_str,recording_number,eyes,parse_ok,parse_notes,file_path,file_name,parent_dir,drug,drug_source,drug_order_confidence,passes_basic_checks,check_notes
0,210,20161207,3,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,210_20161207_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,NaN
1,210,20161207,7,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,210_20161207_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,NaN
2,219,20161117,3,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,219_20161117_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,NaN
3,219,20161117,7,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,219_20161117_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,NaN
4,249,20161208,3,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,249_20161208_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,NaN
5,249,20161208,7,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,249_20161208_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,NaN
6,251,20170124,3,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,251_20170124_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,NaN
7,251,20170124,7,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,251_20170124_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,NaN
8,265,20170112,4,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,265_20170112_0004_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,NaN
9,265,20170112,7,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,265_20170112_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,NaN


In [4]:
# %%
# ============================================
# Section 3. Helpers for QC and epoch-level feature extraction
# ============================================

def pick_region_indices(ch_names: list[str]) -> dict[str, list[int]]:
    """
    Region definition based on channel name prefixes.
    Works reasonably for standard 10-20 style names like Fp1, F3, Cz, Pz, O1, T7, etc.
    If your naming is different, adjust here.
    """
    regions = {"frontal": [], "central": [], "parietal": [], "occipital": [], "temporal": []}

    for i, ch in enumerate(ch_names):
        name = ch.upper()

        # Exclude non-EEG channels if present
        if any(x in name for x in ["EOG", "ECG", "EMG", "AUX", "TRIG", "STI"]):
            continue

        if name.startswith(("FP", "AF", "F")):
            regions["frontal"].append(i)
        elif name.startswith(("C",)):
            regions["central"].append(i)
        elif name.startswith(("P",)):
            regions["parietal"].append(i)
        elif name.startswith(("O",)):
            regions["occipital"].append(i)
        elif name.startswith(("T",)):
            regions["temporal"].append(i)

    # Remove empty regions to avoid NaNs later
    regions = {k: v for k, v in regions.items() if len(v) > 0}
    return regions


def peak_to_peak_uv(epoch_data_volts: np.ndarray) -> float:
    """
    epoch_data_volts: shape (n_channels, n_times)
    returns max peak-to-peak amplitude across channels in microvolts
    """
    ptp_per_ch = np.ptp(epoch_data_volts, axis=1)  # volts
    return float(np.max(ptp_per_ch) * 1e6)


def compute_epoch_ptp_uv(epochs: mne.Epochs) -> np.ndarray:
    """
    Returns ptp per epoch (max across channels) in microvolts.
    Shape: (n_epochs,)
    """
    data = epochs.get_data()  # (n_epochs, n_channels, n_times), volts
    ptps = np.zeros(data.shape[0], dtype=float)
    for e in range(data.shape[0]):
        ptps[e] = peak_to_peak_uv(data[e])
    return ptps


def bandpower_epochwise(
    epochs: mne.Epochs,
    bands_hz: dict[str, tuple[float, float]],
) -> dict[str, np.ndarray]:
    """
    Compute bandpower per epoch per channel.
    Returns dict band -> array shape (n_epochs, n_channels)

    Uses Welch PSD over the full epoch duration (n_fft = n_times).
    """
    data = epochs.get_data()  # (n_epochs, n_channels, n_times), volts
    sfreq = float(epochs.info["sfreq"])
    n_times = data.shape[-1]

    psds, freqs = mne.time_frequency.psd_array_welch(
        data,
        sfreq=sfreq,
        fmin=min(b[0] for b in bands_hz.values()),
        fmax=max(b[1] for b in bands_hz.values()),
        n_fft=n_times,
        n_overlap=0,
        verbose="ERROR",
    )

    out: dict[str, np.ndarray] = {}
    for band_name, (fmin, fmax) in bands_hz.items():
        idx = np.where((freqs >= fmin) & (freqs < fmax))[0]
        if len(idx) == 0:
            raise RuntimeError(f"No frequency bins for band {band_name}")

        bp = np.mean(psds[..., idx], axis=-1)  # (n_epochs, n_channels)
        out[band_name] = bp

    return out


def summarize_bandpower_features_epochwise(
    epochs: mne.Epochs,
    bands_hz: dict[str, tuple[float, float]],
) -> pd.DataFrame:
    """
    Returns epoch-level features (one row per epoch):
    - log bandpower global mean across channels (per epoch)
    - log bandpower per region mean across channels in that region (per epoch)
    """
    regions = pick_region_indices(epochs.ch_names)
    bp = bandpower_epochwise(epochs, bands_hz)

    n_epochs = len(epochs)
    out = pd.DataFrame({"epoch_index_within_clean": np.arange(n_epochs, dtype=int)})

    # For each band: compute per-epoch channel-mean log-power
    for band, bp_ec in bp.items():  # (n_epochs, n_channels)
        bp_ec = np.maximum(bp_ec, 1e-20)

        # Global per epoch: mean across channels
        out[f"logbp_{band}_global"] = np.log(bp_ec).mean(axis=1)

        # Regions per epoch: mean across channels in region
        for region, idxs in regions.items():
            out[f"logbp_{band}_{region}"] = np.log(bp_ec[:, idxs]).mean(axis=1)

    # Region availability metadata
    out["n_regions"] = float(len(regions))
    return out

In [5]:
# %%
# ============================================
# Section 4. Extract Feature Set A PER EPOCH for all EC-only recordings
# Conceptually:
# - For each file:
#   - load epochs
#   - compute ptp per epoch and apply minimal epoch rejection
#   - compute bandpower for kept epochs
#   - store one row per kept epoch
# - Store metadata needed for later analysis and debugging.
# ============================================

rows = []

for _, r in df.iterrows():
    sid = str(r["subject_id"])
    fp = r["file_path"]
    drug = r["drug"]
    eyes = r["eyes"]
    recnum = int(r["recording_number"])

    try:
        epochs = mne.io.read_epochs_eeglab(fp, verbose="ERROR")
        epochs.load_data()  # need data in memory for QC and PSD

        sfreq = float(epochs.info["sfreq"])
        n_epochs_before = len(epochs)
        n_channels = int(epochs.info["nchan"])
        epoch_len_sec = epochs.get_data().shape[-1] / sfreq

        # Compute per-epoch ptp and keep mask
        ptp_uv = compute_epoch_ptp_uv(epochs)  # (n_epochs_before,)
        keep_mask = ptp_uv <= REJECT_PTP_UV

        # If everything got rejected, fail fast for this file
        if int(keep_mask.sum()) == 0:
            raise RuntimeError("All epochs rejected by peak-to-peak threshold")

        # Keep only good epochs
        good_epoch_indices = np.where(keep_mask)[0].astype(int)  # indices in ORIGINAL epochs
        epochs_clean = epochs[keep_mask]
        n_epochs_after = len(epochs_clean)

        # Epoch-level bandpower features for kept epochs
        feat_df = summarize_bandpower_features_epochwise(epochs_clean, BANDS_HZ)

        # Attach original epoch indices + per-epoch ptp
        feat_df["epoch_index_original"] = good_epoch_indices
        feat_df["ptp_uv"] = ptp_uv[good_epoch_indices]

        # Attach recording-level metadata (repeated for each epoch row)
        feat_df["subject_id"] = sid
        feat_df["drug"] = drug
        feat_df["eyes"] = eyes
        feat_df["recording_number"] = recnum
        feat_df["file_path"] = fp
        feat_df["sfreq"] = sfreq
        feat_df["n_channels"] = n_channels
        feat_df["epoch_len_sec"] = epoch_len_sec
        feat_df["n_epochs_before"] = n_epochs_before
        feat_df["n_epochs_after"] = n_epochs_after
        feat_df["extract_ok"] = True
        feat_df["extract_error"] = ""

        rows.append(feat_df)

    except Exception as e:
        # On failure, add a single row indicating error at recording-level
        rows.append(pd.DataFrame([{
            "subject_id": sid,
            "drug": drug,
            "eyes": eyes,
            "recording_number": recnum,
            "file_path": fp,
            "sfreq": np.nan,
            "n_channels": np.nan,
            "epoch_len_sec": np.nan,
            "n_epochs_before": np.nan,
            "n_epochs_after": np.nan,
            "epoch_index_original": np.nan,
            "epoch_index_within_clean": np.nan,
            "ptp_uv": np.nan,
            "extract_ok": False,
            "extract_error": str(e),
        }]))

features_A_epoch = pd.concat(rows, ignore_index=True)

print("Extraction failures (recordings):", features_A_epoch.loc[~features_A_epoch["extract_ok"], "file_path"].nunique())
print("Rows total (epochs + failure rows):", len(features_A_epoch))
features_A_epoch.head(10)

Extraction failures (recordings): 0
Rows total (epochs + failure rows): 276


,epoch_index_within_clean,logbp_delta_global,logbp_delta_frontal,logbp_delta_central,logbp_delta_parietal,logbp_delta_occipital,logbp_delta_temporal,logbp_theta_global,logbp_theta_frontal,logbp_theta_central,...,eyes,recording_number,file_path,sfreq,n_channels,epoch_len_sec,n_epochs_before,n_epochs_after,extract_ok,extract_error
0,0,-26.385868,-26.308092,-27.062983,-26.006362,-25.453731,-26.595514,-26.447130,-26.719572,-26.755095,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
1,1,-26.349598,-26.294731,-27.039179,-25.861005,-25.431104,-26.667982,-25.461217,-25.613886,-25.990038,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
2,2,-26.501530,-26.492941,-27.032717,-26.092004,-25.965406,-26.590647,-26.122721,-26.276793,-26.655976,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
3,3,-26.419617,-26.474607,-26.895923,-26.006024,-25.442976,-26.685923,-25.907130,-25.897162,-26.700442,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
4,4,-26.546696,-26.499935,-27.013993,-26.276086,-26.069918,-26.524552,-25.964807,-25.921989,-26.603458,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
5,5,-27.059936,-26.937633,-27.542114,-26.805443,-26.423926,-27.370621,-26.346549,-26.273498,-27.177611,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
6,6,-26.539647,-26.373201,-27.008827,-26.329823,-25.936640,-27.050877,-26.677044,-26.621358,-27.334780,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
7,7,-26.927507,-26.894730,-27.252723,-26.744155,-26.227754,-27.202671,-26.874448,-26.934714,-27.439365,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
8,8,-26.761068,-26.650867,-27.301632,-26.482901,-26.125847,-27.006747,-26.659117,-26.765430,-27.177101,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,
9,9,-27.002467,-26.842738,-27.547068,-26.806188,-26.296583,-27.281604,-26.631106,-26.603054,-27.299263,...,closed,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,14,14,True,


In [6]:
# %%
# ============================================
# Section 5. QC summary for epoch-level Feature Set A
# ============================================

ok = features_A_epoch[features_A_epoch["extract_ok"]].copy()

print("Epoch-rows ok:", len(ok))
print("Subjects ok:", ok["subject_id"].nunique())
print("Recordings ok:", ok[["subject_id", "recording_number", "file_path"]].drop_duplicates().shape[0])

print("Epoch rows per drug:")
display(ok.groupby("drug").size().rename("n_epochs"))

print("Epoch ptp summary (kept epochs):")
display(ok["ptp_uv"].describe())

print("Sampling rates:")
display(ok.groupby("sfreq").size().rename("n_epochs"))

print("Channel counts:")
display(ok.groupby("n_channels").size().rename("n_epochs"))

# Rejection rate per recording (computed from repeated metadata)
rec_qc = (
    ok[["subject_id", "drug", "recording_number", "file_path", "n_epochs_before", "n_epochs_after"]]
    .drop_duplicates()
    .copy()
)
rec_qc["epoch_drop_frac"] = 1.0 - (rec_qc["n_epochs_after"] / rec_qc["n_epochs_before"])
print("Epoch drop fraction per recording summary:")
display(rec_qc["epoch_drop_frac"].describe())

rec_qc.sort_values("epoch_drop_frac", ascending=False).head(10)[
    ["subject_id", "drug", "recording_number", "n_epochs_before", "n_epochs_after", "epoch_drop_frac", "file_path"]
]

Epoch-rows ok: 276
Subjects ok: 10
Recordings ok: 20
Epoch rows per drug:


drug
awake       141
ketamine    135
Name: n_epochs, dtype: int64

Epoch ptp summary (kept epochs):


count    276.000000
mean      83.143884
std       31.109441
min       29.439360
25%       60.774950
50%       79.024408
75%      100.599679
max      247.219322
Name: ptp_uv, dtype: float64

Sampling rates:


...

Channel counts:


n_channels
62    276
Name: n_epochs, dtype: int64

Epoch drop fraction per recording summary:


count    20.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: epoch_drop_frac, dtype: float64

,subject_id,drug,recording_number,n_epochs_before,n_epochs_after,epoch_drop_frac,file_path
0,210,awake,3,14,14,0.0,/Users/I743312/Documents/ketamine project/data...
14,210,ketamine,7,11,11,0.0,/Users/I743312/Documents/ketamine project/data...
248,318,awake,3,14,14,0.0,/Users/I743312/Documents/ketamine project/data...
234,313,ketamine,7,14,14,0.0,/Users/I743312/Documents/ketamine project/data...
219,313,awake,3,15,15,0.0,/Users/I743312/Documents/ketamine project/data...
205,300,ketamine,7,14,14,0.0,/Users/I743312/Documents/ketamine project/data...
191,300,awake,3,14,14,0.0,/Users/I743312/Documents/ketamine project/data...
180,282,ketamine,7,11,11,0.0,/Users/I743312/Documents/ketamine project/data...
166,282,awake,3,14,14,0.0,/Users/I743312/Documents/ketamine project/data...
152,271,ketamine,7,14,14,0.0,/Users/I743312/Documents/ketamine project/data...


In [8]:
# %%
# ============================================
# Section 6. Save epoch-level Feature Set A
# ============================================
# %%

# Put identifiers/labels first for readability
id_cols = ["subject_id", "drug", "eyes", "recording_number", "epoch_index_original", "epoch_index_within_clean"]
meta_cols = ["sfreq", "n_channels", "epoch_len_sec", "ptp_uv", "n_epochs_before", "n_epochs_after", "file_path", "extract_ok", "extract_error"]
feat_cols = [c for c in features_A_epoch.columns if c.startswith("logbp_") or c == "n_regions"]

ordered = [c for c in id_cols + meta_cols + feat_cols if c in features_A_epoch.columns] + \
          [c for c in features_A_epoch.columns if c not in (id_cols + meta_cols + feat_cols)]

features_A_epoch = features_A_epoch[ordered]
features_A_epoch.to_csv(FEATURES_A_EPOCH_PATH, index=False)

print("Saved:", FEATURES_A_EPOCH_PATH)
print("Shape:", features_A_epoch.shape)

# Optional: save the band definitions used for traceability
meta_path = FEATURE_DIR / "features_A_bandpower_epochwise_meta.json"
meta = {
    "bands_hz": BANDS_HZ,
    "reject_ptp_uv": REJECT_PTP_UV,
    "eyes_keep": EYES_KEEP,
    "feature_file": str(FEATURES_A_EPOCH_PATH),
    "unit_of_analysis": "epoch (one row per kept epoch)",
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved: /Users/I743312/Documents/ketamine project/data/derived/features/features_A_bandpower_epochwise.csv
Shape: (276, 40)
Saved meta: /Users/I743312/Documents/ketamine project/data/derived/features/features_A_bandpower_epochwise_meta.json
